# LangChain: a hands-on implementation

**Track:** Frameworks · **Guide:** [`054-langchain.md`](../guides/concepts/054-langchain.md)

LangChain provides composable integrations and runnable abstractions for building model-powered applications.

> **Intuition:** It is a box of compatible adapters and connectors; the application still owns its contracts and failure behavior.

This lesson builds a small working example in stages: prepare inputs, implement
the core idea, run it on a realistic toy case, and inspect its output and limits.
It runs offline unless a cell is explicitly marked as an optional provider call.

```mermaid
flowchart LR
    A[Prepare inputs] --> B[Implement LangChain]
    B --> C[Run on example]
    C --> D[Evaluate and inspect]
```

**Case study:** Build a small FAQ chain, then test retrieval and model output separately rather than hiding them in one opaque call.

Run the cells from top to bottom. To install the notebook environment, follow
[`guides/SETUP.md`](../guides/SETUP.md). The toy data and implementation are for
learning, not a production quality, safety, or security guarantee.


## 1. Define a composable chain contract

A chain connects retrieval, prompt construction, model invocation, and output parsing. This offline implementation uses a deterministic model adapter.


In [ ]:
documents = [
    {"id": "policy-1", "text": "Unused items may be returned within 30 days."},
    {"id": "policy-2", "text": "Support is open Monday through Friday."},
]

def retrieve(question):
    terms = set(question.lower().split())
    return [doc for doc in documents if terms & set(doc["text"].lower().replace(".", "").split())]

def format_prompt(question, evidence):
    return {"question": question, "context": [doc["text"] for doc in evidence]}

## 2. Compose runnable stages

Each stage accepts structured data and returns structured data; changing the model adapter does not require rewriting retrieval.


In [ ]:
def offline_model(prompt):
    if not prompt["context"]:
        return "I don't know based on the provided evidence."
    return prompt["context"][0]

def parse_answer(text):
    return {"answer": text, "grounded": text != "I don't know based on the provided evidence."}

def run_chain(question):
    evidence = retrieve(question)
    prompt = format_prompt(question, evidence)
    return parse_answer(offline_model(prompt)), evidence

## 3. Execute and inspect intermediate results

The real LangChain version uses prompt templates and runnable composition; see `examples/integrations/langchain_example.py` for an optional provider-backed chain.


In [ ]:
answer, evidence = run_chain("What is the return period?")
print("evidence:", evidence)
print("answer:", answer)
assert answer["grounded"]

## 4. Optionally run the actual LangChain chain

Install the `langchain` extra and set `OPENAI_API_KEY` before opting in. The deterministic chain above remains the default offline path.


In [ ]:
RUN_LANGCHAIN = False
if RUN_LANGCHAIN:
    from examples.integrations.langchain_example import build_chain

    hosted_chain = build_chain()
    hosted_answer = hosted_chain.invoke({
        "question": "What is the return period?",
        "evidence": "Unused items may be returned within 30 days.",
    })
    print(hosted_answer)
else:
    print("Hosted LangChain call skipped; enable it only with credentials and reviewed data.")

## Wrap-up and next steps

- Identify what was learned, stored, retrieved, or generated in this implementation.
- Change an input and observe how the output changes.
- Test one failure case and decide when the system should abstain or ask for help.
- Replace toy data/components only after you can evaluate the change against a
  held-out or representative test set.

Continue with the [concept guide](../guides/concepts/054-langchain.md) and the
[learning roadmap](../guides/ROADMAP.md).
